# Final 2026 NCAA Seed Prediction Notebook (`v8`)

This Colab notebook runs the final private `v8` pipeline end to end.

What it does:
- loads the historical NCAA team-sheet style data
- attaches historical true seeds and historical `AQ / AL / NONE` labels
- validates candidate models with rolling-origin backtests
- uses weekly `2026` snapshots only for stability checks
- selects the final model conservatively
- scores the `2026-03-15` snapshot only
- writes one final CSV in submission format

This notebook follows the same code structure as:
- `code/build_historical_true_seeds.py`
- `code/private_features.py`
- `code/build_submission_v8_final.py`

Important modeling rule:
- `Bid Type` is **not known** in the final `2026-03-15` test snapshot
- we **predict** bid class (`AQ`, `AL`, `NONE`) from pre-selection features
- then enforce the final field composition of `31 AQ` and `37 AL`

## Required files

This notebook expects the repository layout below.

Required raw files:
- `data/raw/NCAA_Seed_Training_Set2.0.csv`
- `data/raw/NCAA_Seed_Test_Set2.0.csv`
- `data/raw/NCAA_Seed_Test_Set_2026_20260206.csv`
- `data/raw/NCAA_Seed_Test_Set_2026_20260208.csv`
- `data/raw/NCAA_Seed_Test_Set_2026_20260215.csv`
- `data/raw/NCAA_Seed_Test_Set_2026_20260222.csv`
- `data/raw/NCAA_Seed_Test_Set_2026_20260301.csv`
- `data/raw/NCAA_Seed_Test_Set_2026_20260308.csv`
- `data/raw/NCAA_Seed_Test_Set_2026_20260315.csv`

Required external label file:
- `data/external/historical_true_seeds_2021_2025.csv`

If the historical true-seed file is missing, the notebook can build it using `build_historical_true_seeds.py`.
If any raw CSV is missing in Colab, upload it into the expected folder before running the evaluation cells.

In [ ]:
# Install the packages used by the v8 pipeline.
!pip -q install pandas numpy scikit-learn lightgbm catboost requests beautifulsoup4 openpyxl

In [ ]:
from pathlib import Path
import os
import sys

# If you cloned the repo in Colab, this is the default location.
DEFAULT_ROOT = Path('/content/final-four-analytics-challenge-26')
PROJECT_ROOT = DEFAULT_ROOT if DEFAULT_ROOT.exists() else Path.cwd()
print('PROJECT_ROOT =', PROJECT_ROOT)

for rel in ['code', 'data/raw', 'data/external', 'artifacts', 'submissions/final']:
    (PROJECT_ROOT / rel).mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / 'code') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'code'))

## Optional: upload missing files in Colab

If you are running this notebook in a fresh Colab session and some CSVs are missing, use the next cell.
It uploads files from your machine into `data/raw/` or `data/external/`.

In [ ]:
# Optional helper for Colab uploads.
# You can skip this cell if the files already exist in the repo/workspace.
required_raw = [
    'NCAA_Seed_Training_Set2.0.csv',
    'NCAA_Seed_Test_Set2.0.csv',
    'NCAA_Seed_Test_Set_2026_20260206.csv',
    'NCAA_Seed_Test_Set_2026_20260208.csv',
    'NCAA_Seed_Test_Set_2026_20260215.csv',
    'NCAA_Seed_Test_Set_2026_20260222.csv',
    'NCAA_Seed_Test_Set_2026_20260301.csv',
    'NCAA_Seed_Test_Set_2026_20260308.csv',
    'NCAA_Seed_Test_Set_2026_20260315.csv',
]
required_external = ['historical_true_seeds_2021_2025.csv']

missing_raw = [name for name in required_raw if not (PROJECT_ROOT / 'data' / 'raw' / name).exists()]
missing_external = [name for name in required_external if not (PROJECT_ROOT / 'data' / 'external' / name).exists()]

print('Missing raw files:', missing_raw)
print('Missing external files:', missing_external)

try:
    from google.colab import files
    if missing_raw or missing_external:
        print('Upload the missing CSV files now. Raw files will be placed in data/raw and historical labels in data/external.')
        uploaded = files.upload()
        for name, data in uploaded.items():
            if name in required_external:
                target = PROJECT_ROOT / 'data' / 'external' / name
            else:
                target = PROJECT_ROOT / 'data' / 'raw' / name
            target.write_bytes(data)
            print('Saved', target)
    else:
        print('All required files are already present.')
except ImportError:
    print('google.colab is not available here. If files are missing, place them in the expected folders manually.')

In [ ]:
# Validate the expected files before importing the pipeline.
required_paths = [
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Training_Set2.0.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set2.0.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set_2026_20260206.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set_2026_20260208.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set_2026_20260215.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set_2026_20260222.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set_2026_20260301.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set_2026_20260308.csv',
    PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set_2026_20260315.csv',
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:
' + '
'.join(missing))
print('All required raw files are present.')

In [ ]:
# Build the historical true-seed file only if it is missing.
# This step uses the checked-in builder script and may require internet access.
import subprocess
import sys

true_seed_path = PROJECT_ROOT / 'data' / 'external' / 'historical_true_seeds_2021_2025.csv'
if not true_seed_path.exists():
    print('Historical true-seed file not found. Building it now...')
    subprocess.run([sys.executable, str(PROJECT_ROOT / 'code' / 'build_historical_true_seeds.py')], check=True)
else:
    print('Historical true-seed file already exists:', true_seed_path)

## Import the final `v8` pipeline

The notebook reuses the project code directly instead of retyping the full pipeline.
That keeps the Colab version aligned with the repository implementation.

In [ ]:
import json
import pandas as pd

from build_submission_v8_final import (
    candidate_specs,
    choose_final_candidate,
    compute_weekly_stability,
    final_train_and_predict,
    load_labeled_historical_universe,
    run_primary_validation,
    run_secondary_loocv,
    summarize_weighted,
    write_submission,
)
from build_submission_v8_final import FINAL_TEST_PATH
from build_submission_v8_final import load_weekly_2026_snapshots

In [ ]:
# Load the labeled historical universe and the final 2026 snapshot.
historical_df = load_labeled_historical_universe(skip_seed_build=False)
final_test_df = pd.read_csv(FINAL_TEST_PATH)
weekly_snapshots = load_weekly_2026_snapshots()

print('Historical universe shape:', historical_df.shape)
print('Final 2026 snapshot shape:', final_test_df.shape)
print('Weekly snapshots:', sorted(weekly_snapshots.keys()))

historical_df[['Season', 'Team', 'TrueSeed', 'BidClass']].head()

## Run rolling-origin historical validation

This is the main selection stage.

Candidate families evaluated:
- `LogisticRegression + RidgeCV`
- `LightGBMClassifier + RidgeCV`
- `LightGBMClassifier + LightGBMRegressor`
- CFA / rank-fusion variant
- seed ensemble variant
- optional CatBoost seed regressor

In [ ]:
# Run the main rolling-origin validation.
candidates = candidate_specs()
primary_metrics, _ = run_primary_validation(historical_df, candidates)
summary = summarize_weighted(primary_metrics[primary_metrics['status'] == 'active'].copy())

artifact_dir = PROJECT_ROOT / 'artifacts' / 'final_20260315_eval_colab'
artifact_dir.mkdir(parents=True, exist_ok=True)
primary_metrics.to_csv(artifact_dir / 'rolling_origin_metrics_v8.csv', index=False)
summary.to_csv(artifact_dir / 'candidate_summary_v8.csv', index=False)

summary

## Secondary validation and weekly stability

We use:
- leave-one-season-out as a robustness check
- weekly `2026` snapshots as a stability check only

The final model is chosen conservatively:
- best weighted rolling RMSE first
- then tie-break by MAE, inclusion quality, stability, and simplicity

In [ ]:
shortlist_names = summary.loc[
    summary['WeightedFullRMSE'] <= summary['WeightedFullRMSE'].min() * 1.01,
    'candidate'
].tolist()
shortlist_candidates = [candidate for candidate in candidates if candidate.name in shortlist_names]

loocv = run_secondary_loocv(historical_df, shortlist_candidates)
weekly_summary, _ = compute_weekly_stability(historical_df, final_test_df, weekly_snapshots, shortlist_candidates)
weekly_agg = weekly_summary.groupby('candidate')[['Top68Jaccard', 'Top16Overlap', 'Top80MeanSeedMovement']].mean().reset_index()
chosen = choose_final_candidate(summary, weekly_agg)

loocv.to_csv(artifact_dir / 'leave_one_season_out_metrics_v8.csv', index=False)
weekly_summary.to_csv(artifact_dir / 'weekly_stability_metrics_v8.csv', index=False)
(artifact_dir / 'model_choice_v8.json').write_text(json.dumps(chosen, indent=2, default=str))

print('Chosen candidate:', chosen['candidate'])
display(summary)
display(weekly_agg)
display(loocv.groupby('candidate')[['FullRMSE', 'FullMAE', 'InclusionF1']].mean().reset_index())
chosen

## Train the final model on all historical seasons and score the 2026-03-15 snapshot

This is the only inference snapshot used for the final submission.

In [ ]:
chosen_name = str(chosen['candidate'])
chosen_candidate = next(candidate for candidate in candidates if candidate.name == chosen_name)

final_scored, final_details = final_train_and_predict(historical_df, final_test_df, chosen_candidate)
output_path = PROJECT_ROOT / 'submissions' / 'final' / 'submission_2026_20260315.csv'
submission = write_submission(final_test_df, final_scored, output_path)

print('Wrote final submission to:', output_path)
submission.head()

In [ ]:
# Save an audit table so we can inspect the selected field and bid composition.
final_audit = final_test_df[['RecordID', 'Season', 'Team', 'Conference', 'NET Rank']].merge(
    final_scored[['RecordID', 'PredictedSeed', 'PredBidType', 'FinalScore', 'P_AQ', 'P_AL', 'AQPriority', 'ALPriority']],
    on='RecordID',
    how='left',
)
final_audit.to_csv(artifact_dir / 'final_selection_audit_v8.csv', index=False)

selected = final_audit[final_audit['PredictedSeed'] > 0].sort_values('PredictedSeed')
print('Selected teams:', len(selected))
print('AQ count:', int((selected['PredBidType'] == 'AQ').sum()))
print('AL count:', int((selected['PredBidType'] == 'AL').sum()))
selected[['Team', 'PredictedSeed', 'PredBidType', 'FinalScore', 'P_AQ', 'P_AL']].head(25)

In [ ]:
# Final submission sanity checks.
assert submission.shape[0] == 365, 'Submission must have 365 rows'
assert submission.columns.tolist() == ['RecordID', 'Overall Seed']
assert (submission['Overall Seed'] > 0).sum() == 68, 'Submission must have exactly 68 seeded teams'
assert set(submission.loc[submission['Overall Seed'] > 0, 'Overall Seed']) == set(range(1, 69)), 'Seed set must be exactly 1..68'
assert submission['RecordID'].tolist() == final_test_df['RecordID'].tolist(), 'RecordID order must match the final test snapshot'
print('All final submission checks passed.')

## Download the final CSV in Colab

If you are running this notebook in Google Colab, the next cell downloads the final submission.

In [ ]:
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print('google.colab is not available here. Final file is saved at:', output_path)